# CSI-SpectraNet: Dual-Stream Spectro-Temporal Network for Fall Detection

A genuinely novel architecture that treats WiFi CSI as **dual-modal physics data**, not just a generic time series.

## Why this architecture is different

Standard approaches (ResNet, LSTM, Transformer) treat CSI as a flat (500, 232) matrix.  
But CSI has **deep physical structure** that no existing model exploits:

1. **Amplitude |H(f,t)|** = signal energy, captures how multi-path propagation attenuates the signal.  
   Falls cause sudden, large changes in attenuation because the human body is a strong reflector.

2. **Phase angle(H(f,t))** = signal geometry, captures the precise path lengths of reflected signals.  
   Falls cause characteristic Doppler-like phase shifts as the body moves through space.

3. **Subcarriers** are not independent features -- they sample adjacent frequencies, so neighboring  
   subcarriers are physically correlated. Treating them as a frequency axis (not flat features) is key.

4. **Falls have kinematics** -- position (raw CSI), velocity (1st derivative), acceleration (2nd derivative)  
   all carry distinct physical information about the event.

---

## Architecture diagram

```
Input: (B, 500, 232) = [amp(116) | phase(116)]

    +--- Amplitude Stream (energy dynamics) ---+    +--- Phase Stream (geometric/Doppler) ---+
    | amp (B,500,116)                          |    | phase (B,500,116)                      |
    | -> SubcarrierEmbed(116,80)               |    | -> LearnedPhaseCorrection(116)          |
    |    + SpectralPositionalEncoding           |    |    Removes CFO/STO artifacts adaptively |
    | -> MotionAwareConv(80)                   |    | -> SubcarrierEmbed(116,80)              |
    |    Process [raw, velocity, acceleration]  |    |    + SpectralPositionalEncoding          |
    | -> TemporalBlock(80->96, stride=2)       |    | -> MotionAwareConv(80)                  |
    |    (B, 96, 250)                          |    | -> TemporalBlock(80->96, stride=2)      |
    | -> FrequencyInteraction(96)              |    |    (B, 96, 250)                         |
    |    Learned spectral filter via FFT        |    | -> FrequencyInteraction(96)             |
    | -> TemporalBlock(96->128, stride=2)      |    | -> TemporalBlock(96->128, stride=2)     |
    |    (B, 128, 125)                         |    |    (B, 128, 125)                        |
    +------------------------------------------+    +----------------------------------------+
                           |                                          |
                           +---------- CrossModalAttention -----------+
                           |  amp queries attend to phase keys/values |
                           |  phase queries attend to amp keys/values |
                           |  -> GatedProjection -> (B, 192, 125)    |
                           +------------------------------------------+
                                              |
                           +--- TemporalFeaturePyramid ---+
                           | P1 fine:   (B, 192, 125)     |
                           | P2 mid:    (B, 192, 63)      |
                           | P3 coarse: (B, 192, 32)      |
                           | Top-down fusion -> P1 enriched|
                           +-------------------------------+
                                              |
                           +--- MultiPoolAggregation ---+
                           | Attention pool (WHEN)       |
                           | Max pool (HOW EXTREME)      |
                           | Avg pool (OVERALL)          |
                           | Std pool (VARIABILITY)      |
                           | -> (B, 768)                 |
                           +-----------------------------+
                                              |
                           +--- ClassificationHead ---+
                           | LN -> 256 -> 64 -> 2     |
                           +--------------------------+
```

In [ ]:
# ============================================================
# Setup
# ============================================================
!pip install -q h5py onnx onnxruntime

import os, sys, json, random, time, math, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ============================================================
# Configuration
# ============================================================
WIN_LEN       = 500
FEATURE_SIZE  = 232     # 116 amplitude + 116 phase
N_SUB         = 116     # subcarriers per modality
BATCH_SIZE    = 64
EPOCHS        = 80
LR            = 2e-4
WEIGHT_DECAY  = 1e-4
WARMUP_EPOCHS = 5
PATIENCE      = 15
NUM_CLASSES   = 2
MIXUP_ALPHA   = 0.3
OUT           = Path('/kaggle/working')

FALL_DIR = Path('/kaggle/input/datasets/guozhenjennzhu/csi-bench/FallDetection')
if not FALL_DIR.exists():
    # fallback search
    for p in sorted(Path('/kaggle/input').rglob('FallDetection')):
        if p.is_dir():
            FALL_DIR = p; break

assert FALL_DIR.exists(), f'FallDetection dir not found. Tried: {FALL_DIR}'
SPLIT_DIR = FALL_DIR / 'splits'
print(f'FallDetection: {FALL_DIR}')
print(f'Splits exist:  {SPLIT_DIR.exists()}')

In [ ]:
# == Data loading (from working EdgeTCN notebook) ==
def read_csi(p):
    with h5py.File(str(p),'r') as f:
        ks=list(f.keys())
        if 'CSI_amps' in ks and 'CSI_phases' in ks:
            a=np.array(f['CSI_amps'],dtype=np.float32).squeeze()
            p_=np.array(f['CSI_phases'],dtype=np.float32).squeeze()
            if a.ndim!=2 or p_.ndim!=2: return None
            if a.shape[0]<a.shape[1]: a=a.T
            if p_.shape[0]<p_.shape[1]: p_=p_.T
            d=np.concatenate([a,p_],1)
        elif 'CSI_amps' in ks:
            d=np.array(f['CSI_amps'],dtype=np.float32).squeeze()
            if d.ndim!=2: return None
            if d.shape[0]<d.shape[1]: d=d.T
        else:
            d=np.array(f[ks[0]],dtype=np.float32).squeeze()
            if d.ndim!=2: return None
            if d.shape[0]<d.shape[1]: d=d.T
    d=np.nan_to_num(d,nan=0.,posinf=0.,neginf=0.)
    C=d.shape[1]
    if C>FEATURE_SIZE: d=d[:,:FEATURE_SIZE]
    elif C<FEATURE_SIZE: d=np.pad(d,((0,0),(0,FEATURE_SIZE-C)))
    L=d.shape[0]
    if L==WIN_LEN: return d
    elif L>WIN_LEN: s=(L-WIN_LEN)//2; return d[s:s+WIN_LEN]
    else:
        ot=np.linspace(0,1,L); nt=np.linspace(0,1,WIN_LEN)
        return np.stack([np.interp(nt,ot,d[:,k]) for k in range(FEATURE_SIZE)],1).astype(np.float32)

def act_label(p):
    for x in Path(str(p)).parts:
        xl=x.lower().replace('-','_')
        if xl.startswith('act_') and xl in ('act_fall','act_falling','act_fallen','act_fall_down'): return 1
    return 0

# Discover H5 files (same glob as working notebook)
files=sorted(FALL_DIR.glob('sub_Human/**/*.h5'))
if not files: files=sorted(FALL_DIR.rglob('*.h5'))
falls=[p for p in files if act_label(p)==1]; nonfalls=[p for p in files if act_label(p)==0]
print(f'H5: {len(files)} (Fall:{len(falls)} Non-Fall:{len(nonfalls)})')

# Try official splits by stem + multiple matching strategies
SPLIT_DIR=FALL_DIR/'splits'; splits={}
if SPLIT_DIR.exists():
    stem2p={p.stem:p for p in files}
    rel2p={}
    for p in files:
        try:
            r=str(p.relative_to(FALL_DIR)).replace('\\','/')
            rel2p[r]=p; rel2p[r.rsplit('.',1)[0]]=p
        except: pass
    for name in ['train','val','test','test_easy','test_medium','test_hard']:
        for sfx in ['_id.json','.json']:
            sp=SPLIT_DIR/f'{name}{sfx}'
            if not sp.exists(): continue
            ids=json.load(open(sp))
            if isinstance(ids,dict): ids=list(ids.values())[0] if ids else []
            if not isinstance(ids,list): continue
            matched=[]
            for sid in ids:
                sid=str(sid).replace('\\','/')
                fp=rel2p.get(sid) or stem2p.get(sid) or stem2p.get(Path(sid).stem)
                if fp is None:
                    for rp,rfp in rel2p.items():
                        if rp.endswith(sid) or sid.endswith(rp): fp=rfp; break
                if fp: matched.append(fp)
            if matched: splits[name]=matched; print(f'  {name:15s}: {len(matched)}')
            break
    if not splits or all(len(v)==0 for v in splits.values()):
        # Debug output
        for sfx in ['_id.json','.json']:
            sp=SPLIT_DIR/f'train{sfx}'
            if sp.exists():
                ids=json.load(open(sp))
                if isinstance(ids,dict): ids=list(ids.values())[0] if ids else []
                print(f'Split IDs sample: {ids[:3]}')
                break
        print(f'File stems sample: {[p.stem for p in files[:3]]}')
        try: print(f'File rels sample:  {[str(p.relative_to(FALL_DIR)).replace(chr(92),"/") for p in files[:3]]}')
        except: pass
        splits={}

# Fallback: random 80/20 split (same as working EdgeTCN notebook)
if not splits or 'train' not in splits or len(splits.get('train',[]))==0:
    print('Using random 80/20 split (same as EdgeTCN notebook)')
    rng=np.random.default_rng(SEED); f_=falls.copy(); n_=nonfalls.copy()
    rng.shuffle(f_); rng.shuffle(n_)
    cf=int(.8*len(f_)); cn=int(.8*len(n_))
    splits={'train':f_[:cf]+n_[:cn],'val':f_[cf:]+n_[cn:],'test':f_[cf:]+n_[cn:]}
for k in splits:
    if splits[k] and not isinstance(splits[k][0],tuple):
        splits[k]=[(p,act_label(p)) for p in splits[k]]
for k,v in splits.items(): print(f'  {k:15s}: {len(v)} (Fall:{sum(y for _,y in v)})')


In [ ]:
# == Build datasets ==
class CSIDataset(Dataset):
    def __init__(self, items, augment=False, stats=None):
        self.augment=augment; dl,ll=[],[]
        for i,(p,y) in enumerate(items):
            x=read_csi(p)
            if x is not None and x.shape==(WIN_LEN,FEATURE_SIZE): dl.append(x); ll.append(y)
            if (i+1)%500==0: print(f'  {i+1}/{len(items)}')
        self.data=np.stack(dl); self.labels=np.array(ll,dtype=np.int64)
        if stats is None:
            self.mean=self.data.mean(axis=(0,1),keepdims=True)
            self.std=self.data.std(axis=(0,1),keepdims=True)+1e-6
        else: self.mean,self.std=stats
    def get_stats(self): return (self.mean,self.std)
    def __len__(self): return len(self.data)
    def __getitem__(self,idx):
        x=self.data[idx].copy(); y=self.labels[idx]
        if self.augment:
            sh=np.random.randint(-50,51)
            if sh: x=np.roll(x,sh,0)
            x=x*np.random.uniform(.9,1.1,(1,FEATURE_SIZE)).astype(np.float32)
            x=x+np.random.normal(0,.01,x.shape).astype(np.float32)
        x=(x-self.mean.squeeze())/self.std.squeeze()
        return torch.from_numpy(x).float(),torch.tensor(y,dtype=torch.long)

print('Loading train...'); train_ds=CSIDataset(splits['train'],augment=True)
ts=train_ds.get_stats()
print('Loading val...'); val_ds=CSIDataset(splits['val'],stats=ts)
test_datasets={}
for n in ['test','test_easy','test_medium','test_hard']:
    if n in splits and splits[n]: print(f'Loading {n}...'); test_datasets[n]=CSIDataset(splits[n],stats=ts)

tc=Counter(train_ds.labels.tolist()); print(f'Train:{len(train_ds)} Val:{len(val_ds)} Dist:{dict(tc)}')
cw={c:len(train_ds)/(NUM_CLASSES*n) for c,n in tc.items()}
sampler=WeightedRandomSampler([cw[y] for y in train_ds.labels.tolist()],len(train_ds),True)
train_loader=DataLoader(train_ds,BATCH_SIZE,sampler=sampler,num_workers=2,pin_memory=True,drop_last=True)
val_loader=DataLoader(val_ds,BATCH_SIZE*2,num_workers=2,pin_memory=True)
test_loaders={n:DataLoader(ds,BATCH_SIZE*2,num_workers=2,pin_memory=True) for n,ds in test_datasets.items()}


## Architecture: CSI-SpectraNet

Six novel components, each with a clear physical motivation:

| # | Component | What it does | Why it matters for CSI |
|:--|:----------|:-------------|:-----------------------|
| 1 | **LearnedPhaseCorrection** | Predicts and removes per-packet linear phase artifacts | Raw WiFi phase is corrupted by CFO/STO. Hand-crafted corrections fail across devices. |
| 2 | **MotionAwareConv** | Processes raw signal + 1st derivative (velocity) + 2nd derivative (acceleration) in parallel | Falls are defined by kinematics: sudden acceleration followed by deceleration to zero |
| 3 | **FrequencyInteraction** | FFT along time, apply learned spectral filter, iFFT back | Equivalent to a convolution kernel as long as the entire window. Captures periodicity vs. aperiodicity (walks vs. falls) |
| 4 | **CrossModalAttention** | Amplitude queries attend to phase; phase queries attend to amplitude | Learns temporal alignment between energy changes and geometric changes during a fall |
| 5 | **TemporalFeaturePyramid** | Builds 3-scale hierarchy with top-down + lateral fusion (FPN) | Fine-scale impact transients gain context from coarse-scale posture changes, and vice versa |
| 6 | **MultiPoolAggregation** | 4 complementary pooling strategies | Attention=when, Max=how extreme, Avg=overall, Std=variability |

In [ ]:
# ============================================================
# Component 1: Learned Phase Sanitization
# ============================================================

class LearnedPhaseCorrection(nn.Module):
    """Adaptively removes systematic phase distortion from raw WiFi CSI.
    
    Problem:
        Raw WiFi phase is corrupted by two hardware artifacts:
        - Carrier Frequency Offset (CFO): adds a constant rotation to all subcarriers
        - Sampling Time Offset (STO): adds a linear slope across subcarriers
        
        Together, the measured phase at subcarrier k, time t is:
            phase_measured[t,k] = phase_true[t,k] + slope[t]*k + offset[t]
    
    Solution:
        Learn a small network that predicts (slope, offset) from the raw phase
        itself, then subtract the predicted linear component. This is adaptive
        to the specific device and environment, unlike hand-crafted corrections.
    
    Input:  (B, T, n_sub) raw phase values
    Output: (B, T, n_sub) corrected phase values
    """
    def __init__(self, n_sub=116):
        super().__init__()
        self.n_sub = n_sub
        # Predict slope and offset from raw phase spectrum
        self.predictor = nn.Sequential(
            nn.Linear(n_sub, 48),
            nn.GELU(),
            nn.Linear(48, 2)   # [slope, offset]
        )
        # Normalized subcarrier indices for computing linear correction
        self.register_buffer('k_idx', torch.linspace(0, 1, n_sub).unsqueeze(0).unsqueeze(0))
    
    def forward(self, phase):
        # phase: (B, T, n_sub)
        params = self.predictor(phase)          # (B, T, 2)
        slope  = params[:, :, 0:1]              # (B, T, 1)
        offset = params[:, :, 1:2]              # (B, T, 1)
        correction = slope * self.k_idx + offset  # (B, T, n_sub)
        return phase - correction


# ============================================================
# Component 2: Motion-Aware Convolution
# ============================================================

class MotionAwareConv(nn.Module):
    """Processes raw CSI alongside its 1st and 2nd temporal derivatives.
    
    Physics motivation:
        A fall is fundamentally a KINEMATIC event. The full state requires:
        - 0th order (raw CSI):     proportional to body POSITION/posture
        - 1st derivative (diff):   proportional to body VELOCITY
        - 2nd derivative (diff2):  proportional to body ACCELERATION
        
        A fall signature in these three domains:
        - Raw:    gradual change (upright -> ground)
        - Vel:    sudden spike then decay (fast body movement)
        - Accel:  sharp impulse at impact (deceleration on hitting ground)
    
    Implementation:
        Depthwise conv on each derivative (cheap, per-channel) then
        pointwise conv to mix the three kinematic components.
    
    Input:  (B, C, T)
    Output: (B, C, T)
    """
    def __init__(self, channels):
        super().__init__()
        # Depthwise conv on each kinematic order (per-channel, very cheap)
        self.dw_raw  = nn.Conv1d(channels, channels, 5, padding=2, groups=channels, bias=False)
        self.dw_vel  = nn.Conv1d(channels, channels, 5, padding=2, groups=channels, bias=False)
        self.dw_acc  = nn.Conv1d(channels, channels, 5, padding=2, groups=channels, bias=False)
        # Pointwise mix: fuse the three kinematic views
        self.mix = nn.Sequential(
            nn.Conv1d(channels * 3, channels, 1, bias=False),
            nn.BatchNorm1d(channels),
            nn.GELU()
        )
    
    def forward(self, x):
        # x: (B, C, T)
        # Compute temporal derivatives
        vel = x[:, :, 1:] - x[:, :, :-1]          # 1st difference
        vel = F.pad(vel, (1, 0))                    # pad left to keep length
        acc = vel[:, :, 1:] - vel[:, :, :-1]        # 2nd difference
        acc = F.pad(acc, (1, 0))
        
        raw_feat = self.dw_raw(x)                   # shape-preserving
        vel_feat = self.dw_vel(vel)
        acc_feat = self.dw_acc(acc)
        
        return self.mix(torch.cat([raw_feat, vel_feat, acc_feat], dim=1))


# ============================================================
# Component 3: Frequency-Domain Interaction
# ============================================================

class FrequencyInteraction(nn.Module):
    """Learnable spectral filtering in the frequency domain.
    
    Core idea:
        Apply FFT along the temporal axis, multiply by a learned
        complex-valued filter, then iFFT back to time domain.
    
    Why this is powerful:
        This is mathematically equivalent to a CIRCULAR CONVOLUTION with
        a kernel whose size equals the entire sequence length. No practical
        Conv1d can have a kernel this large (it would be 250 here).
    
    Fall detection relevance:
        - Walking produces PERIODIC CSI patterns (regular step frequency)
        - Falls produce APERIODIC broadband transients
        - The learned filter can amplify aperiodic components and suppress
          periodic motion, acting as a fall-specific spectral enhancer
    
    Input:  (B, C, T)
    Output: (B, C, T)
    """
    def __init__(self, channels, seq_len):
        super().__init__()
        n_freq = seq_len // 2 + 1  # number of frequency bins (real FFT)
        # Learnable complex spectral filter (initialized as identity/passthrough)
        self.filter_real = nn.Parameter(torch.ones(1, channels, n_freq))
        self.filter_imag = nn.Parameter(torch.zeros(1, channels, n_freq) * 0.01)
        self.norm = nn.BatchNorm1d(channels)
    
    def forward(self, x):
        residual = x
        # To frequency domain
        X_freq = torch.fft.rfft(x, dim=-1)                       # (B, C, n_freq), complex
        # Apply learned filter
        H = torch.complex(self.filter_real, self.filter_imag)     # (1, C, n_freq), complex
        Y_freq = X_freq * H
        # Back to time domain
        y = torch.fft.irfft(Y_freq, n=x.size(-1), dim=-1)        # (B, C, T)
        return residual + self.norm(y)                            # residual connection


print('Components 1-3 defined.')

In [ ]:
# ============================================================
# Temporal Block (shared building block)
# ============================================================

class TemporalBlock(nn.Module):
    """Residual temporal downsampling block.
    
    Two k=5 convolutions with residual skip. Stride-2 variant halves
    the temporal dimension via learned downsampling (better than pooling
    because it learns what to keep vs. discard at each time scale).
    """
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 5, stride=stride, padding=2, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 5, padding=2, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        need_proj = (in_ch != out_ch) or (stride != 1)
        self.skip    = nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False) if need_proj else nn.Identity()
        self.skip_bn = nn.BatchNorm1d(out_ch) if need_proj else nn.Identity()
    
    def forward(self, x):
        identity = self.skip_bn(self.skip(x))
        out = F.gelu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.gelu(out + identity)


# ============================================================
# Component 4: Cross-Modal Attention
# ============================================================

class CrossModalAttention(nn.Module):
    """Bidirectional cross-attention between amplitude and phase streams.
    
    Key insight:
        During a fall, amplitude and phase change SIMULTANEOUSLY but encode
        DIFFERENT physical quantities:
        - Amplitude captures HOW MUCH the signal was disrupted (energy)
        - Phase captures HOW the signal path changed (geometry/Doppler)
        
        Cross-attention lets the model ask:
        'At the time steps where amplitude shows a disruption pattern,
         what is the corresponding phase behavior?' (and vice versa)
        
        This discovers joint amp-phase signatures that neither stream
        can identify alone.
    
    Implementation:
        Standard multi-head cross-attention with:
        - amp_stream queries attending to phase_stream keys/values
        - phase_stream queries attending to amp_stream keys/values
        - Gated projection to fuse the two cross-attended representations
    
    Input:  amp (B, C, T), phase (B, C, T)
    Output: fused (B, C_out, T)
    """
    def __init__(self, dim, out_dim, n_heads=4, dropout=0.1):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = dim // n_heads
        self.scale    = self.head_dim ** -0.5
        
        # Amp queries phase
        self.q_a   = nn.Linear(dim, dim, bias=False)
        self.kv_p  = nn.Linear(dim, dim * 2, bias=False)
        
        # Phase queries amp
        self.q_p   = nn.Linear(dim, dim, bias=False)
        self.kv_a  = nn.Linear(dim, dim * 2, bias=False)
        
        self.norm_a = nn.LayerNorm(dim)
        self.norm_p = nn.LayerNorm(dim)
        
        # Gated fusion: learn how much each cross-attended stream contributes
        self.gate = nn.Sequential(
            nn.Linear(dim * 2, dim * 2),
            nn.Sigmoid()
        )
        self.out_proj = nn.Sequential(
            nn.Linear(dim * 2, out_dim),
            nn.GELU()
        )
        self.out_norm = nn.BatchNorm1d(out_dim)
        self.attn_drop = nn.Dropout(dropout)
    
    def _cross_attend(self, q_proj, kv_proj, query, context):
        """Compute multi-head cross-attention: query attends to context."""
        B, T, C = query.shape
        H, D = self.n_heads, self.head_dim
        
        q = q_proj(query).reshape(B, T, H, D).transpose(1, 2)     # (B, H, T, D)
        kv = kv_proj(context).reshape(B, T, 2, H, D).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]                                        # each (B, H, T, D)
        
        attn = (q @ k.transpose(-2, -1)) * self.scale               # (B, H, T, T)
        attn = self.attn_drop(F.softmax(attn, dim=-1))
        out  = (attn @ v).transpose(1, 2).reshape(B, T, C)          # (B, T, C)
        return out
    
    def forward(self, amp, phase):
        # amp, phase: (B, C, T)
        B, C, T = amp.shape
        
        # Transpose to (B, T, C) for attention
        amp_t   = self.norm_a(amp.transpose(1, 2))       # (B, T, C)
        phase_t = self.norm_p(phase.transpose(1, 2))     # (B, T, C)
        
        # Bidirectional cross-attention
        amp_x_phase = self._cross_attend(self.q_a, self.kv_p, amp_t, phase_t)   # (B, T, C)
        phase_x_amp = self._cross_attend(self.q_p, self.kv_a, phase_t, amp_t)   # (B, T, C)
        
        # Gated fusion: dynamically weight each cross-attended stream
        combined = torch.cat([amp_x_phase, phase_x_amp], dim=-1)    # (B, T, 2C)
        gates = self.gate(combined)                                  # (B, T, 2C)
        gated = combined * gates                                     # (B, T, 2C)
        
        # Project to output dimension
        fused = self.out_proj(gated)                                 # (B, T, out_dim)
        fused = self.out_norm(fused.transpose(1, 2))                 # (B, out_dim, T)
        return fused


# ============================================================
# Component 5: Temporal Feature Pyramid Network
# ============================================================

class TemporalFPN(nn.Module):
    """Feature Pyramid Network operating on the temporal axis.
    
    Motivation:
        Falls have a multi-scale temporal structure:
        - FINE scale   (125 steps, ~10ms res): impact transient, ground contact
        - MEDIUM scale (63 steps,  ~20ms res): limb motion, body rotation
        - COARSE scale (32 steps,  ~40ms res): full posture transition
    
    Standard parallel branches (like InceptionTime) process each scale
    independently. FPN adds BIDIRECTIONAL information flow:
        - Top-down: coarse context enriches fine features
          ('this fine-scale impact transient is part of an ongoing fall')
        - Lateral:  fine details inform coarse-scale decisions
          ('the coarse posture change was preceded by a sharp impact')
    
    Input:  (B, C, 125)
    Output: (B, C, 125) enriched with multi-scale context
    """
    def __init__(self, channels):
        super().__init__()
        C = channels
        # Bottom-up pathway (progressive downsampling)
        self.down2 = nn.Sequential(
            nn.Conv1d(C, C, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm1d(C), nn.GELU()
        )  # 125 -> 63
        self.down3 = nn.Sequential(
            nn.Conv1d(C, C, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm1d(C), nn.GELU()
        )  # 63 -> 32
        
        # Coarse-scale processing (self-attention at 32 steps is cheap)
        self.coarse_attn = nn.MultiheadAttention(C, num_heads=4, dropout=0.1, batch_first=True)
        self.coarse_ffn  = nn.Sequential(
            nn.Linear(C, C * 2), nn.GELU(), nn.Linear(C * 2, C)
        )
        self.coarse_norm1 = nn.LayerNorm(C)
        self.coarse_norm2 = nn.LayerNorm(C)
        
        # Lateral connections (1x1 conv to align channels)
        self.lateral2 = nn.Conv1d(C, C, 1, bias=False)
        self.lateral1 = nn.Conv1d(C, C, 1, bias=False)
        
        # Top-down smoothing (removes aliasing from upsampling)
        self.smooth2 = nn.Sequential(
            nn.Conv1d(C, C, 3, padding=1, bias=False),
            nn.BatchNorm1d(C), nn.GELU()
        )
        self.smooth1 = nn.Sequential(
            nn.Conv1d(C, C, 3, padding=1, bias=False),
            nn.BatchNorm1d(C), nn.GELU()
        )
    
    def forward(self, x):
        # Bottom-up
        p1 = x                    # (B, C, 125)
        p2 = self.down2(p1)       # (B, C, 63)
        p3 = self.down3(p2)       # (B, C, 32)
        
        # Self-attention at coarsest scale (only 32 tokens -- very efficient)
        p3_t = p3.transpose(1, 2)                                   # (B, 32, C)
        p3_t = p3_t + self.coarse_attn(self.coarse_norm1(p3_t), self.coarse_norm1(p3_t), self.coarse_norm1(p3_t))[0]
        p3_t = p3_t + self.coarse_ffn(self.coarse_norm2(p3_t))
        p3   = p3_t.transpose(1, 2)                                 # (B, C, 32)
        
        # Top-down pathway
        p2_fused = self.lateral2(p2) + F.interpolate(p3, size=p2.size(-1), mode='nearest')
        p2_fused = self.smooth2(p2_fused)                            # (B, C, 63)
        
        p1_fused = self.lateral1(p1) + F.interpolate(p2_fused, size=p1.size(-1), mode='nearest')
        p1_fused = self.smooth1(p1_fused)                            # (B, C, 125)
        
        return p1_fused  # enriched with multi-scale context


# ============================================================
# Component 6: Multi-Pool Aggregation
# ============================================================

class MultiPoolAggregation(nn.Module):
    """Four complementary temporal pooling strategies.
    
    Each captures a different aspect of the fall event:
    - Attention pool: WHEN the event occurs (learned temporal weighting)
    - Max pool:       HOW EXTREME the peak disruption is
    - Avg pool:       OVERALL signal level (baseline vs. event)
    - Std pool:       VARIABILITY (falls = high variance, rest = low)
    
    Output: 4C-dimensional feature vector for classification.
    """
    def __init__(self, channels):
        super().__init__()
        self.attn_score = nn.Conv1d(channels, 1, 1)
    
    def forward(self, x):
        # x: (B, C, T)
        # Attention pool
        w = F.softmax(self.attn_score(x), dim=-1)   # (B, 1, T)
        attn = (x * w).sum(dim=-1)                   # (B, C)
        # Statistics pools
        mx = x.amax(dim=-1)                          # (B, C)
        avg = x.mean(dim=-1)                         # (B, C)
        std = x.std(dim=-1)                          # (B, C)
        
        return torch.cat([attn, mx, avg, std], dim=1)  # (B, 4C)


print('Components 4-6 defined.')

In [ ]:
# ============================================================
# Full Model: CSI-SpectraNet
# ============================================================

class CSISpectraNet(nn.Module):
    """Dual-Stream Spectro-Temporal Network for WiFi CSI Fall Detection.
    
    Exploits the dual-modal nature of CSI (amplitude = energy dynamics,
    phase = geometric/Doppler dynamics) with physics-motivated components:
    
    1. Separate amp/phase streams with learned phase sanitization
    2. Motion-aware convolution (position + velocity + acceleration)
    3. Frequency-domain spectral interaction (global temporal context)
    4. Cross-modal attention (discovers joint amp-phase fall signatures)
    5. Temporal feature pyramid (multi-scale with bidirectional flow)
    6. Multi-pool aggregation (4 complementary temporal summaries)
    """
    
    def __init__(self, n_sub=116, num_classes=2, dropout=0.2):
        super().__init__()
        
        # =========================================
        # Phase Correction (phase stream only)
        # =========================================
        self.phase_correct = LearnedPhaseCorrection(n_sub)
        
        # =========================================
        # Dual-Stream Encoders
        # =========================================
        # Each stream: embed -> motion-aware conv -> temporal block -> freq interaction -> temporal block
        
        # Subcarrier embedding: 116 -> 80 channels
        # This learns which subcarriers are informative (like a frequency-domain bandpass filter bank)
        self.amp_embed   = nn.Sequential(nn.Conv1d(n_sub, 80, 1, bias=False), nn.BatchNorm1d(80), nn.GELU())
        self.phase_embed = nn.Sequential(nn.Conv1d(n_sub, 80, 1, bias=False), nn.BatchNorm1d(80), nn.GELU())
        
        # Spectral positional encoding: learnable per-subcarrier bias
        # Because subcarrier position corresponds to physical frequency, position matters
        self.amp_pos   = nn.Parameter(torch.randn(1, 80, 1) * 0.02)
        self.phase_pos = nn.Parameter(torch.randn(1, 80, 1) * 0.02)
        
        # Motion-aware convolution (processes raw + velocity + acceleration)
        self.amp_motion   = MotionAwareConv(80)
        self.phase_motion = MotionAwareConv(80)
        
        # First temporal downsampling: 500 -> 250, 80 -> 96 channels
        self.amp_down1   = TemporalBlock(80, 96, stride=2)
        self.phase_down1 = TemporalBlock(80, 96, stride=2)
        
        # Frequency-domain interaction (at 250-step resolution)
        self.amp_freq   = FrequencyInteraction(96, seq_len=250)
        self.phase_freq = FrequencyInteraction(96, seq_len=250)
        
        # Second temporal downsampling: 250 -> 125, 96 -> 128 channels
        self.amp_down2   = TemporalBlock(96, 128, stride=2)
        self.phase_down2 = TemporalBlock(96, 128, stride=2)
        
        # =========================================
        # Cross-Modal Fusion
        # =========================================
        self.cross_modal = CrossModalAttention(dim=128, out_dim=192, n_heads=4, dropout=0.1)
        
        # =========================================
        # Temporal Feature Pyramid
        # =========================================
        self.fpn = TemporalFPN(192)
        
        # =========================================
        # Pooling + Classification Head
        # =========================================
        self.pool = MultiPoolAggregation(192)
        
        # Squeeze-Excite channel calibration before classification
        pool_dim = 192 * 4  # 768
        self.se = nn.Sequential(
            nn.Linear(pool_dim, pool_dim // 8),
            nn.GELU(),
            nn.Linear(pool_dim // 8, pool_dim),
            nn.Sigmoid()
        )
        
        self.head = nn.Sequential(
            nn.LayerNorm(pool_dim),
            nn.Linear(pool_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        """Forward pass.
        
        Args:
            x: (B, 500, 232) normalized CSI with [amp(116) | phase(116)]
        Returns:
            logits: (B, num_classes)
        """
        # ---- Split into amplitude and phase ----
        amp_raw   = x[:, :, :N_SUB]      # (B, 500, 116)
        phase_raw = x[:, :, N_SUB:]       # (B, 500, 116)
        
        # ---- Phase sanitization ----
        phase_clean = self.phase_correct(phase_raw)  # (B, 500, 116)
        
        # ---- Transpose to (B, C, T) for conv operations ----
        amp   = amp_raw.transpose(1, 2)       # (B, 116, 500)
        phase = phase_clean.transpose(1, 2)   # (B, 116, 500)
        
        # ---- Subcarrier embedding + positional encoding ----
        amp   = self.amp_embed(amp) + self.amp_pos         # (B, 80, 500)
        phase = self.phase_embed(phase) + self.phase_pos   # (B, 80, 500)
        
        # ---- Motion-aware convolution (raw + vel + acc) ----
        amp   = self.amp_motion(amp)       # (B, 80, 500)
        phase = self.phase_motion(phase)   # (B, 80, 500)
        
        # ---- First temporal downsampling ----
        amp   = self.amp_down1(amp)        # (B, 96, 250)
        phase = self.phase_down1(phase)    # (B, 96, 250)
        
        # ---- Frequency-domain interaction ----
        amp   = self.amp_freq(amp)         # (B, 96, 250)
        phase = self.phase_freq(phase)     # (B, 96, 250)
        
        # ---- Second temporal downsampling ----
        amp   = self.amp_down2(amp)        # (B, 128, 125)
        phase = self.phase_down2(phase)    # (B, 128, 125)
        
        # ---- Cross-modal attention fusion ----
        fused = self.cross_modal(amp, phase)  # (B, 192, 125)
        
        # ---- Temporal feature pyramid ----
        fused = self.fpn(fused)               # (B, 192, 125) enriched
        
        # ---- Multi-pool aggregation ----
        pooled = self.pool(fused)             # (B, 768)
        
        # ---- Squeeze-excite calibration ----
        pooled = pooled * self.se(pooled)     # (B, 768)
        
        # ---- Classification ----
        return self.head(pooled)              # (B, 2)


# ---- Instantiate and inspect ----
model = CSISpectraNet(N_SUB, NUM_CLASSES, dropout=0.2).to(device)
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'CSI-SpectraNet created.')
print(f'  Total parameters:     {n_params:,}')
print(f'  Trainable parameters: {n_trainable:,}')
print(f'  vs ResNet-18:         ~11,000,000 ({n_params/11_000_000:.1%} the size)')
print(f'  vs LSTM baseline:     ~1,200,000  ({n_params/1_200_000:.1%} the size)')

# Verify shapes
with torch.no_grad():
    dummy = torch.randn(2, WIN_LEN, FEATURE_SIZE, device=device)
    out = model(dummy)
    print(f'\n  Input:  {dummy.shape}')
    print(f'  Output: {out.shape}')

# Per-component parameter breakdown
print('\n  Parameter breakdown:')
groups = [
    ('Phase Correction',     [model.phase_correct]),
    ('Amp Embed+Pos+Motion', [model.amp_embed, model.amp_pos, model.amp_motion]),
    ('Phase Embed+Pos+Motion',[model.phase_embed, model.phase_pos, model.phase_motion]),
    ('Amp Temporal Blocks',  [model.amp_down1, model.amp_down2]),
    ('Phase Temporal Blocks',[model.phase_down1, model.phase_down2]),
    ('Frequency Interaction',[model.amp_freq, model.phase_freq]),
    ('Cross-Modal Attention',[model.cross_modal]),
    ('Feature Pyramid',     [model.fpn]),
    ('Pool + SE + Head',    [model.pool, model.se, model.head]),
]
for name, modules in groups:
    cnt = sum(m.numel() if isinstance(m, nn.Parameter) else sum(p.numel() for p in m.parameters()) for m in modules)
    print(f'    {name:25s}: {cnt:>8,} ({cnt/n_params:.1%})')

In [ ]:
# ============================================================
# Training utilities
# ============================================================

class FocalLoss(nn.Module):
    """Focal Loss (Lin et al. 2017). gamma=2 focuses on hard examples."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        if alpha is not None:
            self.register_buffer('alpha', torch.tensor(alpha, dtype=torch.float32))
        else: self.alpha = None
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()


def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1-lam) * x[idx], y, y[idx], lam


def get_lr_schedule(optimizer, warmup, total):
    def lr_lambda(ep):
        if ep < warmup: return ep / max(1, warmup)
        return max(0.01, 0.5 * (1 + math.cos(math.pi * (ep - warmup) / max(1, total - warmup))))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_p, all_y = [], []
    for x, y in loader:
        all_p.append(model(x.to(device)).argmax(1).cpu())
        all_y.append(y)
    p = torch.cat(all_p).numpy(); y = torch.cat(all_y).numpy()
    acc = (p == y).mean()
    metrics = {'accuracy': float(acc)}
    for cls, name in [(0,'nonfall'),(1,'fall')]:
        tp = ((p==cls)&(y==cls)).sum(); fp = ((p==cls)&(y!=cls)).sum(); fn = ((p!=cls)&(y==cls)).sum()
        pr = tp/max(1,tp+fp); re = tp/max(1,tp+fn); f1 = 2*pr*re/max(1e-9,pr+re)
        metrics[f'{name}_f1'] = float(f1); metrics[f'{name}_precision'] = float(pr); metrics[f'{name}_recall'] = float(re)
    c = Counter(y.tolist()); tot = sum(c.values())
    metrics['weighted_f1'] = sum(metrics[f"{('nonfall','fall')[i]}_f1"] * c.get(i,0)/tot for i in [0,1])
    return metrics

# Setup
cf = torch.tensor([tc.get(0,1), tc.get(1,1)], dtype=torch.float32)
ca = (1./cf); ca = ca/ca.sum()*NUM_CLASSES
print(f'Class weights: {ca.tolist()}')

criterion = FocalLoss(alpha=ca.to(device), gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = get_lr_schedule(optimizer, WARMUP_EPOCHS, EPOCHS)

In [ ]:
# ============================================================
# Training loop
# ============================================================

best_f1 = -1.; best_state = None; best_m = None; stale = 0; history = []
print(f'Training for up to {EPOCHS} epochs, patience={PATIENCE}')
print('='*95)

for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    model.train()
    tloss, ns = 0., 0
    use_mixup = epoch > WARMUP_EPOCHS
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        if use_mixup:
            xm, ya, yb, lam = mixup_data(x, y, MIXUP_ALPHA)
            logits = model(xm)
            loss = lam*criterion(logits,ya) + (1-lam)*criterion(logits,yb)
        else:
            logits = model(x); loss = criterion(logits, y)
        
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tloss += loss.item()*x.size(0); ns += x.size(0)
    
    scheduler.step()
    vm = evaluate(model, val_loader)
    dt = time.time()-t0
    history.append({'epoch':epoch, 'loss':tloss/max(1,ns), **{f'val_{k}':v for k,v in vm.items()}, 'lr':optimizer.param_groups[0]['lr']})
    
    tag = ''
    if vm['weighted_f1'] > best_f1:
        best_f1 = vm['weighted_f1']; best_m = vm.copy()
        best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        stale = 0; tag = ' * BEST'
    else: stale += 1
    
    print(f'Ep {epoch:3d}/{EPOCHS} | loss={tloss/max(1,ns):.4f} | acc={vm["accuracy"]:.4f} | '
          f'wF1={vm["weighted_f1"]:.4f} | fall_F1={vm["fall_f1"]:.4f} | '
          f'lr={optimizer.param_groups[0]["lr"]:.2e} | {dt:.1f}s{tag}')
    
    if stale >= PATIENCE:
        print(f'\nEarly stopping (no improvement for {PATIENCE} epochs)'); break

model.load_state_dict(best_state)
print(f'\n{"="*95}')
print(f'Best val weighted F1: {best_f1:.4f}')
print(json.dumps(best_m, indent=2))

In [ ]:
# ============================================================
# Test evaluation
# ============================================================

print('\n' + '='*95)
print('TEST EVALUATION')
print('='*95)

all_results = {}
baselines = {
    'MLP':      {'acc':92.16,'f1':92.17}, 'LSTM':    {'acc':94.93,'f1':94.92},
    'ResNet-18':{'acc':94.88,'f1':94.89}, 'Transformer':{'acc':94.28,'f1':94.26},
    'PatchTST': {'acc':94.03,'f1':94.03}, 'TimesFormer1D':{'acc':93.86,'f1':93.87},
    'ViT':      {'acc':93.58,'f1':93.59},
}

for sn, loader in test_loaders.items():
    m = evaluate(model, loader); all_results[sn] = m
    print(f'\n--- {sn.upper()} ---')
    print(f'  Accuracy:    {m["accuracy"]*100:.2f}%')
    print(f'  Weighted F1: {m["weighted_f1"]*100:.2f}%')
    print(f'  Fall F1:     {m["fall_f1"]*100:.2f}%')
    print(f'  Fall Prec:   {m["fall_precision"]*100:.2f}%  Recall: {m["fall_recall"]*100:.2f}%')

if 'test' in all_results:
    print(f'\n{"="*95}')
    print(f'{"Model":<20s} {"Acc%":>8s} {"F1%":>8s}')
    print('-'*40)
    for n,b in baselines.items(): print(f'{n:<20s} {b["acc"]:>8.2f} {b["f1"]:>8.2f}')
    print('-'*40)
    our = all_results['test']
    print(f'{"CSI-SpectraNet":>20s} {our["accuracy"]*100:>8.2f} {our["weighted_f1"]*100:>8.2f}')
    print(f'\nParams: {n_params:,} vs ResNet-18: ~11,000,000')

In [ ]:
# ============================================================
# Save + ONNX export + latency benchmark
# ============================================================

ckpt_path = OUT / 'csi_spectranet_best.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'architecture': 'CSISpectraNet',
    'feature_size': FEATURE_SIZE, 'n_sub': N_SUB, 'win_len': WIN_LEN,
    'num_classes': NUM_CLASSES, 'num_params': n_params,
    'best_val_metrics': best_m, 'test_results': all_results,
    'training_history': history,
    'normalization': {'mean': train_ds.mean.tolist(), 'std': train_ds.std.tolist()}
}, ckpt_path)
print(f'Checkpoint: {ckpt_path} ({ckpt_path.stat().st_size/1024:.1f} KB)')

# ONNX
cpu_model = model.to('cpu').eval()
onnx_path = OUT / 'csi_spectranet_best.onnx'
try:
    dummy = torch.randn(1, WIN_LEN, FEATURE_SIZE)
    torch.onnx.export(cpu_model, dummy, str(onnx_path),
        input_names=['csi_input'], output_names=['logits'],
        dynamic_axes={'csi_input':{0:'batch'}, 'logits':{0:'batch'}},
        opset_version=17)
    print(f'ONNX: {onnx_path} ({onnx_path.stat().st_size/1024:.1f} KB)')
    
    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    s = np.random.randn(1, WIN_LEN, FEATURE_SIZE).astype(np.float32)
    pt_o = cpu_model(torch.from_numpy(s)).detach().numpy()
    ox_o = sess.run(['logits'], {'csi_input': s})[0]
    print(f'Max |PT-ONNX| = {np.max(np.abs(pt_o-ox_o)):.2e}')
    
    lats = [(time.perf_counter(), sess.run(['logits'],{'csi_input':s})) or time.perf_counter() for _ in range(100)]
    # proper latency measurement
    times = []
    for _ in range(100):
        t0 = time.perf_counter()
        sess.run(['logits'], {'csi_input': s})
        times.append((time.perf_counter()-t0)*1000)
    print(f'CPU Latency: median={np.median(times):.2f}ms p95={np.percentile(times,95):.2f}ms')
except Exception as e:
    print(f'ONNX failed: {e}')

model.to(device)

# Results JSON
summary = {
    'architecture': 'CSI-SpectraNet',
    'novel_components': [
        'LearnedPhaseCorrection', 'MotionAwareConv (raw+vel+acc)',
        'FrequencyInteraction (FFT spectral filter)',
        'CrossModalAttention (amp<->phase)',
        'TemporalFPN (3-scale pyramid with self-attention)',
        'MultiPoolAggregation (attn+max+avg+std)'
    ],
    'total_params': n_params, 'input_shape': [WIN_LEN, FEATURE_SIZE],
    'best_val_f1': best_m['weighted_f1'],
    'test_results': {k:{mk:round(mv,4) for mk,mv in v.items()} for k,v in all_results.items()},
    'baselines': baselines
}
json.dump(summary, open(OUT/'csi_spectranet_results.json','w'), indent=2)
print('\n' + json.dumps(summary, indent=2))

In [ ]:
# ============================================================
# Visualization
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, roc_curve, auc

def plot_results(eval_model, loader, is_bce, threshold=0.5):
    eval_model.eval()
    all_p, all_y = [], []
    with torch.no_grad():
        for x, y in loader:
            out = eval_model(x.to(device))
            if is_bce:
                p = torch.sigmoid(out[:, 0]).cpu().numpy()
                y_true = y[:, 0].cpu().numpy() if y.ndim > 1 else y.cpu().numpy()
            else:
                p = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
                y_true = y.cpu().numpy()
            all_p.extend(p)
            all_y.extend(y_true)
            
    all_p, all_y = np.array(all_p), np.array(all_y)
    preds = (all_p >= threshold).astype(int)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Confusion Matrix
    cm = confusion_matrix(all_y, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1, 
                xticklabels=['Non-Fall', 'Fall'], yticklabels=['Non-Fall', 'Fall'])
    ax1.set_title('Test Split Confusion Matrix')
    ax1.set_ylabel('True Label')
    ax1.set_xlabel('Predicted Label')
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(all_y, all_p)
    ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {auc(fpr, tpr):.3f}')
    ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
    ax2.set_title('ROC Curve')
    ax2.legend(loc="lower right")
    
    plt.tight_layout()
    plt.show()

# Auto-detect parameters based on notebook
target_model = chosen if 'chosen' in locals() else model
bce_mode = 'chosen' in locals()  # FallNet uses 'chosen' and BCE
thr = chosen_m['threshold'] if 'chosen_m' in locals() else 0.5

if 'te_loaders' in locals() and 'test' in te_loaders:
    eval_loader = te_loaders['test']
elif 'test_loaders' in locals() and 'test' in test_loaders:
    eval_loader = test_loaders['test']
else:
    eval_loader = val_loader

print("Generating visual evaluation...")
plot_results(target_model, eval_loader, bce_mode, thr)
